# Default configuration


In [ ]:
# Deployment parameters
silver_lakehouse_name = "{SILVER_LAKEHOUSE_NAME}"
silver_schema_name = "{SILVER_SCHEMA_NAME}"
gold_lakehouse_name = "{GOLD_LAKEHOUSE_NAME}"
gold_schema_name = "{GOLD_SCHEMA_NAME}"


In [ ]:
from pyspark.sql import DataFrame, SparkSession, Window
from pyspark.sql.functions import *
from pyspark.sql.types import *
from abc import ABC, abstractmethod
from datetime import datetime, timezone
from typing import Dict, List, Optional, Tuple
import concurrent.futures
import hashlib
import json
import logging

spark = SparkSession.builder.getOrCreate()

# Education Gold is deterministic: UTC everywhere, no silent numeric coercion.
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.ansi.enabled", True)
spark.conf.set("spark.sql.caseSensitive", True)

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# One publication record per validated Gold refresh.
PUBLICATION_TABLE_NAME = "GoldPublication"

# Bumped whenever the transformation changes in a way that alters its output for
# unchanged inputs. It is part of the snapshot fingerprint, so a logic change produces a
# new snapshot identity even when every Silver version is the same.
TRANSFORMATION_VERSION = "1.3.0"

# The targets one refresh publishes together, in dependency order.
GOLD_TARGETS = [
    "DimSchool", "DimStudent", "DimTeacher", "DimSection", "DimContent",
    "DimAssessment", "DimWindow", "DimDate", "BridgeStudentSection",
    "BridgeTeacherSection", "FactAttendanceDay", "FactAssessment",
    "FactSubmissionState", "FactSubmission", "FactStudentAction", "FactEnrollment",
    "AggSchoolAttendance",
]

# Early-warning windows, counted in the learner's own instructional days. They are window
# lengths only: how many absences in the window count as meeting a rule is chosen by the
# reader in the report, so Gold never decides who is flagged.
ATTENDANCE_RULE_WINDOW_DAYS = 20
FOLLOW_UP_WINDOW_DAYS = 10
# A coursework follow-up is compared on the pieces of work in that class due either side of it.
FOLLOW_UP_WORK_PIECES = 5

# Normalized attendance statuses produced by Silver.
ATTENDANCE_PRESENT = "present"
ATTENDANCE_ABSENT = "absent"
ATTENDANCE_EXCUSED = "excused"
ATTENDANCE_KNOWN = (ATTENDANCE_PRESENT, ATTENDANCE_ABSENT, ATTENDANCE_EXCUSED)


# Global functions


In [ ]:
def get_full_table_name(lakehouse: str, table_name: str, schema: str = "") -> str:
    """Three-part name for a schema-enabled Lakehouse, two-part otherwise."""
    return f"{lakehouse}.{schema}.{table_name}" if schema else f"{lakehouse}.{table_name}"

def get_silver_full_table_name(table_name: str) -> str:
    return get_full_table_name(silver_lakehouse_name, table_name, silver_schema_name)

def get_gold_full_table_name(table_name: str) -> str:
    return get_full_table_name(gold_lakehouse_name, table_name, gold_schema_name)

def get_silver_table(table_name: str) -> DataFrame:
    """Read a Silver table through its pinned view. Gold reads Silver and never writes to it."""
    return spark.read.table(get_silver_view_name(table_name))


def get_silver_view_name(table_name: str) -> str:
    return f"pinned_silver_{table_name.lower()}"


_silver_catalog: Dict[str, str] = {}


def get_silver_physical_name(table_name: str) -> str:
    """
    Resolve a Silver table to the casing the catalog actually uses.

    This session enables spark.sql.caseSensitive. A SELECT still resolves a table
    case-insensitively, but DESCRIBE HISTORY does not, and Silver publishes its tables in
    lower case while this notebook refers to them in Pascal case.
    """
    global _silver_catalog
    if not _silver_catalog:
        _silver_catalog = {
            row["tableName"].lower(): row["tableName"]
            for row in spark.sql(f"SHOW TABLES IN {silver_lakehouse_name}").collect()
        }
    actual_name = _silver_catalog.get(table_name.lower(), table_name)
    return get_full_table_name(silver_lakehouse_name, actual_name, silver_schema_name)


def pin_silver_snapshot(table_names: List[str]) -> Dict[str, int]:
    """
    Capture each Silver table's current Delta version and expose it as a pinned view.

    Every later read goes through these views, so one run sees a single consistent set of
    input versions even if Silver is rewritten while the transformation is running.
    The returned versions are what the snapshot fingerprint is built from.

    The view is registered from a DataFrame rather than written as
    `CREATE TEMP VIEW ... VERSION AS OF`: that SQL form is accepted but cannot be read
    back, because the stored view body is re-resolved without the attached Lakehouse and
    the source table then cannot be found.
    """
    versions: Dict[str, int] = {}
    for table_name in table_names:
        full_table_name = get_silver_physical_name(table_name)
        version = int(spark.sql(f"DESCRIBE HISTORY {full_table_name} LIMIT 1").collect()[0]["version"])
        versions[table_name] = version
        (
            spark.read.format("delta")
                 .option("versionAsOf", version)
                 .table(full_table_name)
                 .createOrReplaceTempView(get_silver_view_name(table_name))
        )
    logging.info(f"✅ Pinned {len(versions)} Silver table(s) to their current Delta versions.")
    return versions


def discard_unpublished_snapshots(table_names: List[str]) -> None:
    """
    Clear any candidate left behind by an interrupted run, before staging a new one.

    Keeps only the snapshot the newest publication row names, so the physical tables hold
    the published snapshot plus at most the candidate currently being built.
    """
    published_snapshot_id = get_published_snapshot_id()
    for table_name in table_names:
        full_table_name = get_gold_full_table_name(table_name)
        if not table_exists(full_table_name):
            continue
        if published_snapshot_id:
            spark.sql(f"DELETE FROM {full_table_name} WHERE snapshot_id <> '{published_snapshot_id}'")
        else:
            spark.sql(f"DELETE FROM {full_table_name}")
    if published_snapshot_id:
        logging.info(f"✅ Cleared unpublished candidates; published snapshot {published_snapshot_id[:12]} retained.")
    else:
        logging.info("✅ No published snapshot yet; cleared any partial candidate.")


def get_candidate_view_name(table_name: str) -> str:
    return f"candidate_{table_name.lower()}"


def register_candidate_views(table_names: List[str], snapshot_id: str) -> None:
    """
    Expose only the candidate snapshot's rows for validation.

    The previously published snapshot stays in the same physical table until the new one
    is published, so validation must never see it. Registered from a DataFrame for the
    same reason as the pinned Silver views: a SQL view body is re-resolved on read and
    can fail to find the source table.
    """
    for table_name in table_names:
        (
            spark.table(get_gold_full_table_name(table_name))
                 .where(col("snapshot_id") == lit(snapshot_id))
                 .createOrReplaceTempView(get_candidate_view_name(table_name))
        )


def discard_candidate_snapshot(table_names: List[str], snapshot_id: str) -> None:
    """
    Remove a candidate snapshot that failed to build or validate.

    The previously published snapshot is untouched, so the newest publication row keeps
    describing a complete physical dataset.
    """
    for table_name in table_names:
        full_table_name = get_gold_full_table_name(table_name)
        if table_exists(full_table_name):
            spark.sql(f"DELETE FROM {full_table_name} WHERE snapshot_id = '{snapshot_id}'")
    logging.warning(f"Discarded candidate snapshot {snapshot_id[:12]}; the published snapshot is unchanged.")


def retire_superseded_snapshots(table_names: List[str], published_snapshot_id: str) -> None:
    """
    Drop snapshots other than the published one, after publication has succeeded.

    Running this only after the pointer moves is what makes the switch safe: until then
    both snapshots exist, and a failure leaves the old one complete.
    """
    for table_name in table_names:
        full_table_name = get_gold_full_table_name(table_name)
        spark.sql(f"DELETE FROM {full_table_name} WHERE snapshot_id <> '{published_snapshot_id}'")
    logging.info(f"✅ Retired snapshots superseded by {published_snapshot_id[:12]}.")


def get_published_snapshot_id() -> Optional[str]:
    """The snapshot named by the newest publication row, or None before the first refresh."""
    full_table_name = get_gold_full_table_name(PUBLICATION_TABLE_NAME)
    if not table_exists(full_table_name):
        return None
    rows = spark.sql(
        f"SELECT snapshot_id FROM {full_table_name} ORDER BY published_at DESC LIMIT 1"
    ).collect()
    return rows[0]["snapshot_id"] if rows else None

def get_gold_table(table_name: str) -> DataFrame:
    return spark.read.table(get_gold_full_table_name(table_name))

def table_exists(full_table_name: str) -> bool:
    return spark.catalog.tableExists(full_table_name)

def get_table_counts(full_table_names: Dict[str, str]) -> DataFrame:
    """Row counts per table, with the error kept instead of a silent zero."""
    results: List[Tuple[str, Optional[int], Optional[str]]] = []

    for table, full_table_name in full_table_names.items():
        try:
            results.append((table, spark.table(full_table_name).count(), None))
        except Exception as e:
            results.append((table, None, str(e)))

    schema = StructType([
        StructField("Table", StringType(), True),
        StructField("RecordCount", LongType(), True),
        StructField("Error", StringType(), True),
    ])
    return spark.createDataFrame(results, schema=schema)


# Data quality functions

Silver is validated before any Gold table is written, and Gold is validated before the
refresh is published. A failed check raises so the pipeline stops instead of publishing
a partially refreshed model.


In [ ]:
class GoldDataQualityError(Exception):
    """Raised when an input or output contract is violated."""

def assert_no_rows(label: str, sql: str):
    """The query must return zero rows. Sample rows are logged for triage."""
    offending = spark.sql(sql)
    count = offending.count()
    if count > 0:
        sample = offending.limit(5).toJSON().collect()
        logging.error(f"⛔ {label}: {count} offending row(s). Sample: {sample}")
        raise GoldDataQualityError(f"{label}: {count} offending row(s).")
    logging.info(f"✅ {label}.")

def assert_unique_key(full_table_name: str, key_columns: List[str]):
    keys = ", ".join(key_columns)
    assert_no_rows(
        f"{full_table_name} unique on ({keys})",
        f"SELECT {keys} FROM {full_table_name} GROUP BY {keys} HAVING COUNT(*) > 1",
    )

def assert_not_null(full_table_name: str, columns: List[str]):
    predicate = " OR ".join(f"{c} IS NULL" for c in columns)
    assert_no_rows(
        f"{full_table_name} required columns {columns}",
        f"SELECT * FROM {full_table_name} WHERE {predicate}",
    )

def assert_foreign_key(full_table_name: str, column: str, parent_table_name: str, parent_column: str):
    assert_no_rows(
        f"{full_table_name}.{column} → {parent_table_name}.{parent_column}",
        f"""
        SELECT c.{column}
        FROM {full_table_name} AS c
        WHERE c.{column} IS NOT NULL
          AND NOT EXISTS (SELECT 1 FROM {parent_table_name} AS p WHERE p.{parent_column} = c.{column})
        """,
    )

def assert_valid_intervals(full_table_name: str, key_columns: List[str], primary_key: str):
    """Half-open [valid_from, valid_to) intervals that never overlap for the same member."""
    assert_no_rows(
        f"{full_table_name} non-empty intervals",
        f"SELECT * FROM {full_table_name} WHERE valid_from >= valid_to",
    )
    predicate = " AND ".join(f"a.{c} = b.{c}" for c in key_columns)
    assert_no_rows(
        f"{full_table_name} non-overlapping intervals",
        f"""
        SELECT a.{primary_key}
        FROM {full_table_name} AS a
        JOIN {full_table_name} AS b
          ON {predicate} AND a.{primary_key} < b.{primary_key}
        WHERE a.valid_from < b.valid_to AND b.valid_from < a.valid_to
        """,
    )

def assert_row_count(full_table_name: str, expected: int):
    actual = spark.table(full_table_name).count()
    if actual != expected:
        raise GoldDataQualityError(f"{full_table_name}: expected {expected} row(s), found {actual}.")
    logging.info(f"✅ {full_table_name} has the expected {actual} row(s).")


# Data model functions

`BaseDataModel` follows the Nonprofit Data Solutions pattern: every table declares an
explicit `StructType`, and `create_entities` creates the empty Delta tables concurrently.
The `primaryKey` metadata is descriptive documentation, not an enforced constraint, so
the data quality functions above validate keys explicitly.


In [ ]:
class BaseDataModel(ABC):
    def __init__(self, spark: SparkSession, lakehouse: str, schema: str = ""):
        self.spark = spark
        self.lakehouse = lakehouse
        self.schema = schema

    @abstractmethod
    def get_entities(self) -> List[Tuple[StructType, str, bool]]:
        """Must return a list of (schema, table_name, enable_change_data_feed) tuples."""
        pass

    def create_entities(self):
        """Create all tables concurrently; every failure is reported."""
        entities = self.get_entities()

        with concurrent.futures.ThreadPoolExecutor() as executor:
            futures = [
                executor.submit(self.create_table, schema, table_name, enable_change_data_feed)
                for schema, table_name, enable_change_data_feed in entities
            ]
            concurrent.futures.wait(futures)

            errors = []
            for future in futures:
                try:
                    future.result()
                except Exception as e:
                    errors.append(e)

            if errors:
                error_msgs = "\n".join(str(e) for e in errors)
                raise Exception(f"One or more table creations failed:\n{error_msgs}")

    def create_table(self, schema: StructType, table_name: str, enable_change_data_feed: bool = False):
        full_table_name = get_full_table_name(self.lakehouse, table_name, self.schema)

        try:
            df: DataFrame = self.spark.createDataFrame(data=[], schema=schema)

            if table_exists(full_table_name):
                # Additive nullable columns only; the enrichment notebook fails on a
                # type change rather than silently migrating a published table.
                df.write.format("delta") \
                        .mode("append") \
                        .option("mergeSchema", "true") \
                        .saveAsTable(full_table_name)
                logging.info(f"✅ Table {full_table_name} updated successfully.")
            else:
                df.write.option("delta.enableChangeDataFeed", "true" if enable_change_data_feed else "false") \
                        .format("delta") \
                        .mode("overwrite") \
                        .saveAsTable(full_table_name)
                logging.info(f"✅ Table {full_table_name} created successfully (EnableChangeDataFeed={enable_change_data_feed}).")

        except Exception as e:
            logging.error(f"⛔ Failed to create or append data to table {full_table_name}: {e}")
            raise


# Silver to Gold functions

The model is refreshed as a whole from the current Silver tables, following the Nonprofit
Data Solutions datamart pattern. Every target is recomputed so that corrected dimensions,
changed memberships and deleted source rows are reflected.


In [ ]:
def get_snapshot_id(transformation_version: str, period_start: str, period_end: str, silver_versions: Dict[str, int]) -> str:
    """
    Fingerprint of what actually produced this refresh: the transformation version, the
    reporting period, and the exact Delta version of every Silver input.

    The run cutoff is deliberately excluded. Hashing the clock made every retry a new
    identity even when nothing had changed, and it described none of the inputs, so two
    ids could not be compared to answer "is this the same data".
    """
    payload = json.dumps(
        {
            "transformation_version": transformation_version,
            "period_start": period_start,
            "period_end": period_end,
            "silver_versions": dict(sorted(silver_versions.items())),
        },
        separators=(",", ":"),
        sort_keys=True,
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def write_gold_table(df: DataFrame, table_name: str, snapshot_id: str, as_of: str) -> int:
    """
    Stamp the refresh identity onto every row and append this refresh as a candidate snapshot.

    Overwriting each target in turn left the model mixed whenever a later target failed:
    the earlier tables already held the new snapshot while the later ones still held the
    previous one, and the newest publication row no longer described a complete physical
    dataset. Appending keeps the published snapshot intact and readable until the
    candidate has been validated and published.
    """
    full_table_name = get_gold_full_table_name(table_name)

    stamped = (
        df.withColumn("snapshot_id", lit(snapshot_id))
          .withColumn("as_of", to_timestamp(lit(as_of)))
    )

    expected_columns = spark.table(full_table_name).columns
    missing = [c for c in expected_columns if c not in stamped.columns]
    unexpected = [c for c in stamped.columns if c not in expected_columns]
    if missing or unexpected:
        raise GoldDataQualityError(
            f"{full_table_name}: column mismatch. Missing={missing}, unexpected={unexpected}."
        )

    stamped = stamped.select(*expected_columns)

    # A retry of the same snapshot replaces its own partial rows and nothing else.
    spark.sql(f"DELETE FROM {full_table_name} WHERE snapshot_id = '{snapshot_id}'")
    stamped.write.mode("append").format("delta").saveAsTable(full_table_name)

    written = spark.sql(
        f"SELECT COUNT(*) FROM {full_table_name} WHERE snapshot_id = '{snapshot_id}'"
    ).collect()[0][0]

    # Expose the staged rows immediately, so anything derived from this target reads the
    # candidate rather than the physical table. While a refresh is staged the physical
    # table holds the published snapshot as well, and aggregating it unfiltered silently
    # doubles every measure.
    register_candidate_views([table_name], snapshot_id)

    logging.info(f"✅ {full_table_name} staged {written} candidate row(s).")
    return written

def publish_gold_refresh(snapshot_id: str, as_of: str, period_start: str, period_end: str, tables: List[str]):
    """
    Move the published pointer to this snapshot.

    Written only after every target is refreshed and validated, so consumers can tell a
    complete refresh from an interrupted one. Counts are scoped to the snapshot being
    published, because the superseded one is still physically present at this moment.
    """
    full_table_name = get_gold_full_table_name(PUBLICATION_TABLE_NAME)
    counts = {
        table: spark.sql(
            f"SELECT COUNT(*) FROM {get_gold_full_table_name(table)} WHERE snapshot_id = '{snapshot_id}'"
        ).collect()[0][0]
        for table in tables
    }

    row = [(
        snapshot_id,
        datetime.strptime(as_of, "%Y-%m-%dT%H:%M:%SZ"),
        datetime.strptime(period_start, "%Y-%m-%d").date(),
        datetime.strptime(period_end, "%Y-%m-%d").date(),
        silver_lakehouse_name,
        json.dumps(counts, sort_keys=True, separators=(",", ":")),
        datetime.now(timezone.utc).replace(tzinfo=None),
    )]

    spark.createDataFrame(row, spark.table(full_table_name).schema) \
         .write.mode("append").format("delta").saveAsTable(full_table_name)

    logging.info(f"✅ Published Gold refresh {snapshot_id} covering {len(tables)} table(s).")


def assert_only_published_snapshot(table_names: List[str], published_snapshot_id: str) -> None:
    """Post-publication proof that every table now holds exactly the published snapshot."""
    for table_name in table_names:
        assert_no_rows(
            f"{get_gold_full_table_name(table_name)} holds only snapshot {published_snapshot_id[:12]}",
            f"SELECT * FROM {get_gold_full_table_name(table_name)} WHERE snapshot_id <> '{published_snapshot_id}'",
        )
